# exp-038 — weight-space repair of under-decomposition at depth (ADR 0042)

**What this is.** A separate track *outside* the few-shot pool study. Target: the exp-026 pool-only LoRA fine-tune of Mistral-7B-Instruct-v0.3 (merged, fp16), which stops planning too early on 3- and 4-hop questions. Find the direction in the residual stream that marks a premature stop, remove it from the weights, and test the edit against a length-matched EOS-bias control, a random direction and a norm-only sanity check on the pinned 600.

**Runtime.** A100 40 GB recommended (L4 24 GB works, slower). Not a T4: the fp16 model alone needs ~14.4 GB at eval (exp-026's measured peak). Estimate (not measured): ~3 h on an A100 end to end; see `experiments/exp-038/README.md`.

**Leakage guard.** Directions, layers, alphas, the length control and the final choice are found on MuSiQue **train** rows only (minus the adapter's 2,000 training rows), asserted disjoint from the whole dev file. The pinned 600 are touched only in stage 7, after `choice.json` is committed (exactly once) and the specs are frozen.

**Claude's role** (approved by Jahid on 2026-11-27, ADR 0042). Two pause points. At each, the files named in the cell are on Drive; Jahid shows them to Claude in his terminal; Claude records its choice in `experiments/exp-038/choices.md` (+ `candidates_override.json` / `choice.json`), commits and pushes; the next cell pulls it.

Every stage writes to `/content/exp038`; sweep and confirm items are mirrored to Drive one by one, the other stages after they finish. Done-files are keyed to the content hash of the candidate or spec they ran, so a revised one is re-run, never reused.

## 0. Drive, settings

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BRANCH = 'main'   # exp-038 merged to main; each stage records the commit it ran at (environment.json)
REPO_URL = 'https://github.com/AhmadiJahid/Thesis---QAv2.git'   # public repo: no token in the clone
TARBALL = '/content/drive/MyDrive/thesis-backup.tar.gz'
TARBALL_SHA256 = '33c6072f0e4e46ab94f40f804463c512f439823b80c3e0e9dec0ee1023717d94'  # recorded when the backup was made (2026-11-27)
WORK = '/content/work'                      # <parent>: holds the clone and thesis-qav2-data side by side (docs/compute.md)
REPO = f'{WORK}/Thesis---QAv2'
DATA = f'{WORK}/thesis-qav2-data'
OUT = '/content/exp038'                     # fast local disk
DRIVE_OUT = '/content/drive/MyDrive/thesis-exp038'   # mirror, survives a disconnect
import os, subprocess, json, hashlib, shutil, pathlib, time
os.makedirs(OUT, exist_ok=True); os.makedirs(DRIVE_OUT, exist_ok=True)

_SECRETS = []   # values that must never appear in a printed command or an error message

def sh(cmd, cwd=None):
    if any(s and s in cmd for s in _SECRETS):
        raise RuntimeError('refusing to run a command that contains a secret (not printed)')
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, cwd=cwd)
    if r.returncode != 0:
        raise RuntimeError(f'failed rc={r.returncode}: {cmd}')

def sync():
    # Mirror OUT to Drive (incremental: only new/changed files are copied).
    sh(f'rsync -a "{OUT}/" "{DRIVE_OUT}/"')

def record_commit(stage):
    # The commit the notebook runs at, recorded at start and after every pull.
    sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
    path = f'{OUT}/environment.json'
    env = json.load(open(path)) if os.path.exists(path) else {}
    env.setdefault('commits', []).append({'stage': stage, 'commit': sha, 'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())})
    json.dump(env, open(path, 'w'), indent=1)
    return sha

# Resume after a disconnect: pull back what an earlier session already finished.
if os.listdir(DRIVE_OUT):
    sh(f'rsync -a "{DRIVE_OUT}/" "{OUT}/"')
print(sorted(os.listdir(OUT)))

## 1. Clone main, install pinned dependencies

In [ ]:
# Hugging Face token with access to mistralai/Mistral-7B-Instruct-v0.3: Colab secrets, HF_TOKEN.
# It goes into the environment only; no command below contains it.
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
_SECRETS.append(os.environ['HF_TOKEN'])
os.makedirs(WORK, exist_ok=True)
if not os.path.exists(REPO):
    sh(f'git clone --branch {BRANCH} {REPO_URL} {REPO}')
else:
    sh('git pull --ff-only', cwd=REPO)
assert subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip() == '', 'dirty tree'
SHA = record_commit('start')
print('commit', SHA)

In [ ]:
# requirements.txt pins torch==2.10.0; Colab ships its own CUDA torch, which is kept (recorded below)
# rather than reinstalled. Everything else is installed at the pinned versions.
sh(f"grep -v '^torch==' {REPO}/requirements.txt > /content/req_no_torch.txt")
sh('pip install -q -r /content/req_no_torch.txt')
import torch, transformers, peft
print('torch', torch.__version__, 'transformers', transformers.__version__, 'peft', peft.__version__)
env = json.load(open(f'{OUT}/environment.json'))
env.update({'torch': torch.__version__, 'transformers': transformers.__version__, 'peft': peft.__version__})
json.dump(env, open(f'{OUT}/environment.json', 'w'), indent=1)

## 2. Data setup from the Drive backup

Layout per `docs/compute.md` "Dataset location": `<parent>/thesis-qav2-data/musique/...` next to the clone, and the run artifacts this track needs under the clone's `runs/`. Only the files exp-038 reads are extracted; member names are treated as untrusted (absolute names, `..` and anything resolving outside its root are refused; only regular files are written). If a shared `scripts/colab_setup_data.py` has landed on main by the time you run this, it does the same job; this cell stays self-contained.

In [ ]:
import sys, tarfile
sys.path.insert(0, f'{REPO}/src')
from weight_space import safe_member_path
NEED_DATA = {  # tarball member -> path under DATA
    'Thesis---QA/MusiQue/Data/musique_ans_v1.0_train.jsonl': 'musique/musique_ans_v1.0_train.jsonl',
    'Thesis---QA/MusiQue/Data/musique_ans_v1.0_dev.jsonl': 'musique/musique_ans_v1.0_dev.jsonl',
}
NEED_DATA_PREFIX = {'Thesis---QA/MusiQue/Data/dev_data/': 'musique/dev_data/'}
NEED_RUNS_PREFIX = {  # tarball prefix -> path under REPO
    'Thesis---QAv2/runs/exp-026/20260920_024535/adapter/': 'runs/exp-026/20260920_024535/adapter/',
    'Thesis---QAv2/runs/pool_sweep/pools/size2000_imbalanced_trial0_poolseed42/': 'runs/pool_sweep/pools/size2000_imbalanced_trial0_poolseed42/',
}
NEED_RUNS_FILE = {
    'Thesis---QAv2/runs/exp-014/retrieval/s2_pinned600_imbalanced_rerank_top5.jsonl': 'runs/exp-014/retrieval/s2_pinned600_imbalanced_rerank_top5.jsonl',
    'Thesis---QAv2/runs/exp-026/eval/eval_per_item.json': 'runs/exp-026/eval/eval_per_item.json',
}

def destination(name):
    if name in NEED_DATA:
        return safe_member_path(DATA, NEED_DATA[name])
    if name in NEED_RUNS_FILE:
        return safe_member_path(REPO, NEED_RUNS_FILE[name])
    for pre, to in NEED_DATA_PREFIX.items():
        if name.startswith(pre):
            return safe_member_path(DATA, to + name[len(pre):])
    for pre, to in NEED_RUNS_PREFIX.items():
        if name.startswith(pre):
            return safe_member_path(REPO, to + name[len(pre):])
    return None

marker = f'{DATA}/.exp038_extracted'
if not os.path.exists(marker):
    local_tar = '/content/thesis-backup.tar.gz'
    if not os.path.exists(local_tar):
        shutil.copy(TARBALL, local_tar)
    h = hashlib.sha256()
    with open(local_tar, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 24), b''):
            h.update(chunk)
    assert h.hexdigest() == TARBALL_SHA256, h.hexdigest()
    n = 0
    with tarfile.open(local_tar, 'r:gz') as tar:
        for m in tar:
            dest = destination(m.name)
            if dest is None:
                continue
            if not m.isfile():   # no links, devices or directories from the archive
                continue
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            with tar.extractfile(m) as src, open(dest, 'wb') as dst:
                shutil.copyfileobj(src, dst)
            n += 1
    print('extracted', n, 'files')
    os.remove(local_tar)
    open(marker, 'w').write('ok')
for p in [f'{DATA}/musique/musique_ans_v1.0_train.jsonl', f'{DATA}/musique/dev_data/musique_ans_v1.0_dev_clean.jsonl',
          f'{REPO}/runs/exp-026/20260920_024535/adapter/adapter_model.safetensors',
          f'{REPO}/runs/exp-014/retrieval/s2_pinned600_imbalanced_rerank_top5.jsonl']:
    assert os.path.exists(p), p
print('data OK')

## 3. GPU check

In [ ]:
sh('nvidia-smi')
props = torch.cuda.get_device_properties(0)
gib = props.total_memory / 2**30
print(props.name, f'{gib:.1f} GiB')
assert gib >= 22, 'needs an L4 (24 GB) or A100 (40 GB); a T4 cannot hold the fp16 7B model plus generation'
json.dump({'gpu': props.name, 'total_gib': round(gib, 1)}, open(f'{OUT}/gpu.json', 'w'))
WSR = f'python components/decomposer/weight_space_repair.py --out {OUT} --mirror {DRIVE_OUT}'
CFG = json.load(open(f'{REPO}/configs/exp038_weight_space.json'))

## 4. Stage 1 — splits (train only) and the leakage checks

The split is deterministic. The expected id hashes below were produced on the laptop (CPU); a mismatch means the data or the code differs and the run stops.

In [ ]:
EXPECTED = {'locate': 'c0550bb2044ee834eed2e830fd94e1cfdf6b726932597c4fd93507d93480f929',
            'val_select': '4dd3116b2929949c46ccaaa072a88308933a371d191cec0a8684e73ebbfbde49',
            'val_confirm': '3898a87f080e8271b780afcc7ab8f7a067b282f974def2af1a91f4249eb33105'}
if not os.path.exists(f'{OUT}/splits.json'):
    sh(f'{WSR} split', cwd=REPO)
rec = json.load(open(f'{OUT}/splits.json'))['record']
got = {k: v['ids_sha256'] for k, v in rec['splits'].items()}
assert got == EXPECTED, got
assert all(c['overlap'] == 0 for c in rec['leakage_checks'])
print(len(rec['leakage_checks']), 'leakage checks, all zero overlap'); sync()

## 5. Stage 2 — locate (GPU)

Greedy-generates locate (1,200), val_select (300) and val_confirm (300) with the merged, unedited model and captures the residual stream after every decoder layer at every step boundary of locate and val_select. Estimate: ~15–25 min on an A100.

In [ ]:
sh(f'{WSR} locate', cwd=REPO)
for s in ('locate', 'val_select', 'val_confirm'):
    d = json.load(open(f'{OUT}/locate_{s}.done.json'))
    print(s, d['boundary_labels'], d['skipped_items'], json.dumps(d['baseline_summary']['hop4']))
sync()

## 6. Stage 3 — directions (CPU) and the first pause

In [ ]:
sh(f'{WSR} directions', cwd=REPO)
print(open(f'{OUT}/directions_table.md').read())
sync()

### PAUSE 1 — Claude inspects the directions

Show Claude (in your terminal) `directions_table.md`, `directions_summary.json`, `candidates.json`, `locate_locate.done.json` and `locate_val_select.done.json` (all in `MyDrive/thesis-exp038/`). `locate_val_confirm.done.json` is **not** shown: val_confirm is never used for choosing. Claude decides whether the proposed sweep is right (layers, direction kinds, alphas, EOS biases), writes its reasoning in `experiments/exp-038/choices.md`, and — only if it changes the sweep — commits `experiments/exp-038/candidates_override.json`, then pushes the branch. Run the next cell after the push.

In [ ]:
sh('git pull --ff-only', cwd=REPO)
record_commit('after_pause_1')
override = f'{REPO}/experiments/exp-038/candidates_override.json'
CANDS = override if os.path.exists(override) else f'{OUT}/candidates.json'
print('sweep candidates from', CANDS)
print(open(f'{REPO}/experiments/exp-038/choices.md').read()[-3000:])

## 7. Stage 4 — sweep on val_select (GPU), then the default rule with length matching

`choose` applies ADR 0042's default rule and picks the EOS-bias control whose val_select mean step count is closest to the chosen edit's. If the gap exceeds `sweep.length_match.tolerance_mean_steps`, it writes `refine_candidates.json` (biases inside the bracketing pair) and this cell sweeps those, up to `max_refine_rounds` rounds. The achieved gap is recorded either way.

In [ ]:
sh(f'{WSR} sweep --split val_select --candidates {CANDS}', cwd=REPO)
LM = CFG['sweep']['length_match']
for rnd in range(int(LM['max_refine_rounds']) + 1):
    sh(f'{WSR} choose', cwd=REPO)
    pc = json.load(open(f'{OUT}/proposed_choice.json'))
    lm = pc['length_match']
    print('round', rnd, json.dumps(lm))
    if pc['chosen'] is None or lm.get('within_tolerance') or rnd == int(LM['max_refine_rounds']):
        break
    sh(f'{WSR} sweep --split val_select --candidates {OUT}/refine_candidates.json', cwd=REPO)
print(json.dumps(pc['table'], indent=1))
sync()

### PAUSE 2 — Claude chooses the edit

Show Claude `proposed_choice.json` and `sweep_val_select/*.summary.json`. Claude writes `experiments/exp-038/choice.json`:

```json
{"edit_candidate": "<an orth_* name>", "eos_control_candidate": "<an eos_bias_* name>",
 "chosen_by": "claude-opus-5-5 in Jahid's terminal", "choice_note": "why, in one paragraph"}
```

with the full reasoning in `choices.md` (where it departs from the default rule, or the length gap is outside tolerance, why). Commit **once** and push. A second commit to `choice.json` makes the freeze below refuse: the choice cannot be revised after this point.

In [ ]:
sh('git pull --ff-only', cwd=REPO)
record_commit('after_pause_2')
CHOICE = f'{REPO}/experiments/exp-038/choice.json'
assert os.path.exists(CHOICE), 'commit experiments/exp-038/choice.json first'
commits = subprocess.check_output(['git', 'log', '--format=%H', '--', 'experiments/exp-038/choice.json'], cwd=REPO, text=True).split()
print('choice.json commits:', commits)
if not os.path.exists(f'{OUT}/specs_frozen.json'):
    sh(f'{WSR} make-specs --choice {CHOICE}', cwd=REPO)
sh(f'{WSR} freeze --choice-commits {",".join(commits)}', cwd=REPO)   # refuses unless exactly one commit; never rewrites
sh(f'{WSR} equivalence', cwd=REPO)                                    # stops the run unless the hook and the weight edit agree
sh(f'{WSR} confirm', cwd=REPO)
for p in sorted(pathlib.Path(f'{OUT}/confirm_val_confirm').glob('*.summary.json')):
    d = json.load(open(p)); e = d['edit']
    print(d['name'], json.dumps(d['summary']['overall']), 'delta_fro_ratio', e.get('delta_fro_ratio'))
sync()

## 8. Stage 6 — the pinned 600 (GPU), five arms

Each arm is the exp-026 evaluation command (`unguided`, `--no-few-shot`, fp16, the exp-014 retrieval artifact, seed 42), unbatched, plus `--weight-edit`. Baseline = merged and unedited. Each arm's directory is keyed to its spec's sha256 and the spec is checked against `specs_frozen.json` first. A run that crashed before scoring is moved aside (`decomposer.failed.<time>`) and re-run, so an arm never has two run directories. Estimate: ~15 min per arm on an A100 plus model load.

In [ ]:
ARMS = ['baseline', 'edit', 'eos_control', 'random', 'norm_control']
frozen = json.load(open(f'{OUT}/specs_frozen.json'))
ARM_DIR = {}
for arm in ARMS:
    spec = f'{OUT}/specs/spec_{arm}.json'
    sha = hashlib.sha256(open(spec, 'rb').read()).hexdigest()
    assert sha == frozen['files'][f'spec_{arm}.json'], f'{arm} spec changed after freezing'
    arm_dir = f'{OUT}/pinned600/{arm}.{sha[:12]}'
    ARM_DIR[arm] = arm_dir
    if os.path.exists(f'{arm_dir}/eval/eval_per_item.json'):
        print(arm, 'done'); continue
    dec = pathlib.Path(f'{arm_dir}/decomposer')
    if dec.exists():
        aside = f'{arm_dir}/decomposer.failed.{time.strftime("%Y%m%dT%H%M%S")}'
        print(arm, 'unscored earlier attempt moved to', aside)
        shutil.move(str(dec), aside)
    sh('python components/decomposer/run_decomposer.py --model mistral_7b_instruct --config decomposer_musique.json '
       '--condition unguided --seed 42 --quantization none --adapter runs/exp-026/20260920_024535/adapter --no-few-shot '
       '--retrieval-input runs/exp-014/retrieval/s2_pinned600_imbalanced_rerank_top5.jsonl '
       f'--weight-edit {spec} --output-root {dec}', cwd=REPO)
    runs = sorted(dec.glob('*/results.json'))
    assert len(runs) == 1, f'{arm}: expected one run directory, found {len(runs)}'
    sh(f'python scripts/musique_decompositions_evaluator.py --predictions {runs[0]} --run-dir {arm_dir}/eval --out-prefix eval', cwd=REPO)
    sync()

## 9. Stage 7 — paired tests, overall and per hop, Holm (CPU)

In [ ]:
arms = {a: f'{ARM_DIR[a]}/eval/eval_per_item.json' for a in ARMS}
arms['exp026_stored'] = f'{REPO}/runs/exp-026/eval/eval_per_item.json'
arms['decomposer_metrics'] = {a: str(next(pathlib.Path(f'{ARM_DIR[a]}/decomposer').glob('*/metrics.json'))) for a in ARMS}
json.dump(arms, open(f'{OUT}/arms.json', 'w'), indent=1)
sh(f'python experiments/exp-038/analysis.py --arms {OUT}/arms.json --out {OUT}/analysis', cwd=REPO)
for a in ARMS:
    m = json.load(open(f'{ARM_DIR[a]}/eval/eval_metrics.json'))
    print(a, json.dumps(m.get('decomposition_report_card', {}))[:600])
sync()

## 10. Files to copy back into `experiments/exp-038/` (small; nothing here is data or weights)

In [ ]:
COPY_BACK = ['environment.json', 'gpu.json', 'target.json', 'splits.json', 'locate_locate.done.json', 'locate_val_select.done.json',
             'locate_val_confirm.done.json', 'directions_summary.json', 'directions_table.md', 'candidates.json',
             'refine_candidates.json', 'proposed_choice.json', 'equivalence.json', 'specs_frozen.json', 'arms.json',
             'sweep_val_select/manifest.json', 'analysis/analysis_config.json', 'analysis/analysis_metrics.json']
COPY_BACK += [str(p.relative_to(OUT)) for p in pathlib.Path(OUT).glob('sweep_val_select/*.summary.json')]
COPY_BACK += [str(p.relative_to(OUT)) for p in pathlib.Path(OUT).glob('confirm_val_confirm/*.summary.json')]
COPY_BACK += [str(p.relative_to(OUT)) for p in pathlib.Path(OUT).glob('specs/*.json')]
for a in ARMS:
    COPY_BACK += [str(p.relative_to(OUT)) for p in pathlib.Path(ARM_DIR[a]).glob('decomposer/*/metrics.json')]
    COPY_BACK += [str(p.relative_to(OUT)) for p in pathlib.Path(ARM_DIR[a]).glob('decomposer/*/config.json')]
    COPY_BACK += [str(p.relative_to(OUT)) for p in pathlib.Path(ARM_DIR[a]).glob('eval/eval_metrics.json')]
for f in COPY_BACK:
    print(f'{DRIVE_OUT}/{f}', 'OK' if os.path.exists(f'{OUT}/{f}') else 'MISSING')
print('\nStays on Drive only (too large or data-bearing): acts_*.npy, boundaries_*.jsonl, generations*.jsonl, '
      'specs/*.npy (their sha256 is in the spec files), pinned600/*/decomposer/*/results.json and per-item files.')
sync()